# Figurine Piece-Symbol Mapper

This notebook analyses a chess PDF whose move text uses **figurine notation (FAN)**
and produces a JSON character-mapping compatible with the Flutter app's `fontMap`
parameter in `MoveParser.parse()`.

## How it works

1. **Character inventory** — PyMuPDF scans every page and collects every character
   that appears in a *piece-prefix position* (i.e. before a SAN suffix like `e4`,
   `xf7`, `=Q`, …) but is not a standard SAN or Unicode chess character.

2. **Context labelling** — A Python port of the Dart move-parser simulates the game
   state page-by-page and tries all five piece-letter substitutions at each unknown
   token. When exactly one substitution yields a legal move the mapping is recorded
   with high confidence.

3. **CNN fallback** — Any character not resolved by context is classified by a tiny
   PyTorch CNN trained on:
   - Glyph images rendered directly from the PDF pages at high DPI
   - Synthetic piece images from the five lichess SVG piece sets

4. **JSON export** — The final mapping `{"<char>": "K" | "Q" | "R" | "B" | "N" | ""}` is
   downloaded and pasted into the Flutter app.

## Output

A file `figurine_mapping.json` ready to be loaded into Flutter's `fontMap`.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q pymupdf python-chess torch torchvision Pillow numpy matplotlib scikit-learn requests cairosvg

## Step 2 — Imports & Helper Functions

In [ ]:
import os, re, json, unicodedata
from collections import defaultdict
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor

import fitz                      # PyMuPDF
import chess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageFilter, ImageEnhance
import requests
import cairosvg

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ---------------------------------------------------------------------------
# Character classification helpers
# ---------------------------------------------------------------------------

# Characters that can legitimately START a SAN token — no mapping needed.
STANDARD_SAN_FIRST = frozenset(
    'KQRBNa-hO0-9x=+#-*(){}$!?'
    + '\u2654\u2655\u2656\u2657\u2658\u2659'   # ♔♕♖♗♘♙
    + '\u265a\u265b\u265c\u265d\u265e\u265f'   # ♚♛♜♝♞♟
)

# A SAN suffix starts with a file letter, capture, or promotion.
_SAN_SUFFIX_RE = re.compile(r'^[a-hx=]')
_PURE_NUM_RE   = re.compile(r'^(\d+)(\.+)$')
_COMBINED_RE   = re.compile(r'^(\d+)(\.{1,3})([^.].*)$')
_RESULT_RE     = re.compile(r'^(1-0|0-1|1/2-1/2|\*)$')

# FEN detection (bare FEN anywhere in text — same regex as in Dart).
_BARE_FEN_RE = re.compile(
    r'[rnbqkpRNBQKP1-8]{1,8}(?:/[rnbqkpRNBQKP1-8]{1,8}){7}'
    r'\s+[wb]\s+[KQkq-]+\s+(?:[a-h][36]|-)\s+\d+\s+\d+'
)
_PGN_FEN_RE  = re.compile(r'\[FEN\s+"([^"]+)"\]', re.IGNORECASE)
_PGN_SET_RE  = re.compile(r'\[SetUp\s+"1"\]',    re.IGNORECASE)

def looks_like_san_suffix(s: str) -> bool:
    return bool(_SAN_SUFFIX_RE.match(s)) if s else False


def detect_fen(text: str):
    """Return a FEN string found in text, or None."""
    if _PGN_SET_RE.search(text):
        m = _PGN_FEN_RE.search(text)
        if m:
            return m.group(1).strip()
    m = _PGN_FEN_RE.search(text)
    if m:
        return m.group(1).strip()
    m = _BARE_FEN_RE.search(text)
    if m:
        return re.sub(r'\s+', ' ', m.group(0))
    return None


def normalize_token(s: str) -> str:
    """Apply the same normalisation as Dart's _normaliseToken."""
    # Unicode chess symbols → empty (pawns) or piece letter
    FIG = {'\u2654': 'K', '\u2655': 'Q', '\u2656': 'R', '\u2657': 'B',
           '\u2658': 'N', '\u2659': '',
           '\u265a': 'K', '\u265b': 'Q', '\u265c': 'R', '\u265d': 'B',
           '\u265e': 'N', '\u265f': ''}
    for k, v in FIG.items():
        s = s.replace(k, v)
    s = s.replace('\u00a2', 'c').replace('\u00a3', 'f')   # ¢→c  £→f
    if s.startswith('\\'):
        s = s[1:]
    s = re.sub(r'[).](?=[a-hx=1-8])', '', s)
    s = re.sub(r'[!?]+$', '', s)
    if s.startswith('0-0-0'):
        s = 'O-O-O' + s[5:]
    elif s.startswith('0-0'):
        s = 'O-O' + s[3:]
    return s


def tokenize(text: str) -> list:
    """Tokenise move text, skipping braces/parens/NAGs — port of Dart _tokenise."""
    tokens = []
    i = brace_depth = paren_depth = 0
    n = len(text)
    while i < n:
        c = text[i]
        if c == '{':   brace_depth += 1; i += 1; continue
        if c == '}':   brace_depth = max(0, brace_depth - 1); i += 1; continue
        if brace_depth: i += 1; continue
        if c == '(':   paren_depth += 1; i += 1; continue
        if c == ')':   paren_depth = max(0, paren_depth - 1); i += 1; continue
        if paren_depth: i += 1; continue
        if c in ' \n\r\t': i += 1; continue
        if c == '$':          # NAG code
            i += 1
            while i < n and text[i].isdigit(): i += 1
            continue
        start = i
        while i < n and text[i] not in ' \n\r\t(){}': i += 1
        tokens.append(text[start:i])
    return tokens


print('\u2705 Helpers ready')

## Step 3 — Load the PDF

In [ ]:
from google.colab import drive, files

# Option A: load from Google Drive (recommended for large PDFs)
drive.mount('/content/drive')

# ── SET THIS PATH to your PDF ──────────────────────────────────────────────
PDF_PATH = '/content/drive/MyDrive/chess_books/your_book.pdf'
# ──────────────────────────────────────────────────────────────────────────

# Option B: upload directly (comment out Option A above and uncomment below)
# uploaded = files.upload()
# PDF_PATH = next(iter(uploaded))

doc = fitz.open(PDF_PATH)
print(f'\u2705 Opened: {os.path.basename(PDF_PATH)}')
print(f'   Pages : {len(doc)}')

## Step 4 — Character Inventory

Scan every page for characters that appear in **piece-prefix positions** — i.e. the
first character of a token whose remainder looks like a SAN suffix (`e4`, `xf7`, …).
Characters already in `STANDARD_SAN_FIRST` are skipped.

In [ ]:
# candidate_chars: char → list of (page_num, bbox, font_size)
candidate_chars = defaultdict(list)

# We also keep the full per-page char map for glyph rendering.
# page_char_map: page_num → {char → [(bbox, font_size)]}
page_char_map = {}

MAX_PAGES = len(doc)   # or limit, e.g. min(len(doc), 50)

for page_num in range(MAX_PAGES):
    page = doc[page_num]
    text = page.get_text()
    tokens = tokenize(text)

    # Build char→bbox map from rawdict for this page.
    raw = page.get_text('rawdict', flags=fitz.TEXT_PRESERVE_WHITESPACE)
    char_positions = defaultdict(list)  # char → [(bbox, size)]
    for block in raw.get('blocks', []):
        if block.get('type') != 0:
            continue
        for line in block.get('lines', []):
            for span in line.get('spans', []):
                sz = span.get('size', 12)
                for ch in span.get('chars', []):
                    c = ch.get('c', '')
                    if c and c != '\x00':
                        char_positions[c].append((ch['bbox'], sz))
    page_char_map[page_num] = char_positions

    # Identify candidate piece-prefix characters in the token stream.
    for tok in tokens:
        if not tok or len(tok) < 2:
            continue
        if _PURE_NUM_RE.match(tok) or _RESULT_RE.match(tok):
            continue

        # Handle combined tokens like "22.Ke4" — strip the number part.
        cm = _COMBINED_RE.match(tok)
        effective = cm.group(3) if cm else tok

        first = effective[0]
        rest  = effective[1:]
        if first in STANDARD_SAN_FIRST:
            continue
        if not looks_like_san_suffix(rest):
            continue

        if char_positions.get(first):
            bbox, sz = char_positions[first][0]
            candidate_chars[first].append((page_num, bbox, sz))

print(f'Found {len(candidate_chars)} candidate character(s):')
for ch, occs in sorted(candidate_chars.items(), key=lambda x: -len(x[1])):
    cp = ord(ch)
    name = unicodedata.name(ch, 'UNKNOWN')
    print(f'  U+{cp:04X}  {repr(ch):8s}  {name:40s}  {len(occs):4d} occurrences')

## Step 5 — Render Glyph Images

For each candidate character, crop its appearances from the page at **4× scale** (~288 DPI),
resize to 32×32 grayscale. We keep up to `MAX_GLYPHS_PER_CHAR` samples.

In [ ]:
GLYPH_SIZE    = 32   # pixels (square)
RENDER_SCALE  = 6    # page render scale (6× ≈ 432 DPI — fine detail)
MAX_GLYPHS_PER_CHAR = 30

# glyph_images: char → list of np.float32 arrays shape (32, 32)
glyph_images = {}
# Page pixmap cache to avoid re-rendering the same page multiple times.
_pix_cache = {}

def get_page_pix(page_num):
    if page_num not in _pix_cache:
        page = doc[page_num]
        mat = fitz.Matrix(RENDER_SCALE, RENDER_SCALE)
        _pix_cache[page_num] = page.get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
    return _pix_cache[page_num]


def crop_glyph(page_num, bbox):
    """Crop a character bbox from the scaled page pixmap and return float32 array."""
    pix = get_page_pix(page_num)
    x0, y0, x1, y1 = [v * RENDER_SCALE for v in bbox]
    # Expand by a few pixels so we see the full glyph.
    pad = RENDER_SCALE * 1
    x0, y0 = max(0, x0 - pad), max(0, y0 - pad)
    x1, y1 = min(pix.width, x1 + pad), min(pix.height, y1 + pad)
    if x1 <= x0 or y1 <= y0:
        return None
    clip = fitz.IRect(int(x0), int(y0), int(x1), int(y1))
    try:
        sub_pix = pix.get_pixmap(clip)   # sub-pixmap (fitz >= 1.21)
    except AttributeError:
        # Older fitz: re-render with clip rect
        page = doc[page_num]
        mat = fitz.Matrix(RENDER_SCALE, RENDER_SCALE)
        sub_pix = page.get_pixmap(matrix=mat, clip=fitz.Rect(bbox), colorspace=fitz.csGRAY)
    img = Image.frombytes('L', [sub_pix.width, sub_pix.height], sub_pix.samples)
    img = img.resize((GLYPH_SIZE, GLYPH_SIZE), Image.LANCZOS)
    return np.array(img, dtype=np.float32) / 255.0


print('Rendering glyph images...')
for ch, occs in candidate_chars.items():
    imgs = []
    # Use a spread of occurrences across the document for variety.
    step = max(1, len(occs) // MAX_GLYPHS_PER_CHAR)
    for page_num, bbox, _ in occs[::step][:MAX_GLYPHS_PER_CHAR]:
        g = crop_glyph(page_num, bbox)
        if g is not None:
            imgs.append(g)
    glyph_images[ch] = imgs
    print(f'  {repr(ch):8s}  {len(imgs):3d} glyph images')

print('\u2705 Done')

## Step 6 — Auto-Label via Chess Context

Simulate game state page-by-page (Python-chess).  For each token whose first
character is still unknown, try all five piece-letter substitutions.  When
exactly one yields a legal move, record the mapping with high confidence.

Pawn moves (no piece prefix) and moves already matching the partial mapping are
applied to advance the board normally.

In [ ]:
PIECE_LETTERS = 'KQRBN'

def try_parse_san(board, san):
    try:
        return board.parse_san(san)
    except Exception:
        return None


def auto_label_page(page_text, partial_mapping):
    """
    Parse one page's move text, extending partial_mapping where possible.
    Returns a dict of new {char: piece_letter} mappings found on this page.
    """
    new_map = {}
    combined = {**partial_mapping, **new_map}  # updated lazily below

    # Determine starting position.
    fen = detect_fen(page_text)
    try:
        board = chess.Board(fen) if fen else chess.Board()
    except Exception:
        board = chess.Board()

    def apply_tok(raw_tok):
        """Try to advance board using raw_tok. Returns True on success."""
        nonlocal new_map, combined
        if not raw_tok or len(raw_tok) < 2:
            return False

        combined = {**partial_mapping, **new_map}
        first = raw_tok[0]

        # Apply known mapping.
        if first in combined:
            tok = normalize_token(combined[first] + raw_tok[1:])
        else:
            tok = normalize_token(raw_tok)

        # Try direct parse (covers pawn moves, castling, already-known chars).
        move = try_parse_san(board, tok)
        if move:
            board.push(move)
            return True

        # Unknown first char in piece-prefix position → try substitutions.
        if first not in STANDARD_SAN_FIRST and first not in combined:
            rest = normalize_token(raw_tok[1:])
            if not looks_like_san_suffix(rest):
                return False
            successes = []
            for p in PIECE_LETTERS:
                m = try_parse_san(board, p + rest)
                if m:
                    successes.append((p, m))
            if len(successes) == 1:
                p, move = successes[0]
                new_map[first] = p
                combined = {**partial_mapping, **new_map}
                board.push(move)
                return True

        # Check for pawn-figurine: char + file+rank is a legal pawn move.
        if first not in STANDARD_SAN_FIRST and first not in combined:
            rest = normalize_token(raw_tok[1:])
            m = try_parse_san(board, rest)
            if m and m.promotion is None:  # simple pawn move
                new_map[first] = ''
                combined = {**partial_mapping, **new_map}
                board.push(m)
                return True

        return False

    tokens = tokenize(page_text)
    current_move_num = None
    expect_black = False

    for tok in tokens:
        if _RESULT_RE.match(tok):
            break

        pm = _PURE_NUM_RE.match(tok)
        if pm:
            current_move_num = int(pm.group(1))
            expect_black = len(pm.group(2)) > 1
            continue

        cm = _COMBINED_RE.match(tok)
        if cm:
            num = int(cm.group(1))
            if current_move_num is None or num >= current_move_num - 1:
                current_move_num = num
                expect_black = len(cm.group(2)) > 1
                tok = cm.group(3)   # parse just the move part
            # fall through

        if current_move_num is None:
            continue

        ok = apply_tok(tok)
        if ok:
            expect_black = not expect_black
            if not expect_black:
                current_move_num += 1

    return new_map


# ── Run labeller across all pages ─────────────────────────────────────────
context_mapping = {}   # char → piece_letter  (confirmed by context)
confidence      = {}   # char → count of consistent confirmations

for page_num in range(len(doc)):
    page_text = doc[page_num].get_text()
    new = auto_label_page(page_text, context_mapping)
    for ch, piece in new.items():
        if ch in context_mapping:
            if context_mapping[ch] == piece:
                confidence[ch] = confidence.get(ch, 1) + 1
            else:
                # Conflict — downgrade confidence.
                print(f'  ⚠ Conflict for {repr(ch)}: existing={context_mapping[ch]!r}, new={piece!r}')
        else:
            context_mapping[ch] = piece
            confidence[ch] = 1

print('\nContext-labelled mapping:')
for ch, piece in sorted(context_mapping.items()):
    cp = ord(ch)
    display_piece = piece if piece else '(pawn→empty)'
    print(f'  U+{cp:04X} {repr(ch):8s} → {display_piece:6s}  (confidence: {confidence.get(ch,0)}x)')

unlabelled = [ch for ch in candidate_chars if ch not in context_mapping]
print(f'\nLabelled: {len(context_mapping)}  |  Still unlabelled: {len(unlabelled)}')

## Step 7 — Visual Review

Inspect auto-labels before proceeding.  If a label looks wrong, fix it in
**Step 7b** (manual overrides) and re-run from there.

In [ ]:
PREVIEW_COLS = 10
chars_to_show = sorted(candidate_chars.keys())
n_rows = len(chars_to_show)

if n_rows == 0:
    print('No candidate characters to review.')
else:
    fig, axes = plt.subplots(n_rows, PREVIEW_COLS,
                             figsize=(PREVIEW_COLS * 0.9, n_rows * 1.1))
    if n_rows == 1:
        axes = [axes]

    for row, ch in enumerate(chars_to_show):
        imgs = glyph_images.get(ch, [])
        label = context_mapping.get(ch, '???')
        display_label = repr(label) if label else '\u03b5 (pawn)'
        row_axes = axes[row] if n_rows > 1 else axes[row]

        for col in range(PREVIEW_COLS):
            ax = row_axes[col]
            if col < len(imgs):
                ax.imshow(imgs[col], cmap='gray', vmin=0, vmax=1)
                if col == 0:
                    cp = ord(ch)
                    color = 'green' if ch in context_mapping else 'red'
                    ax.set_ylabel(
                        f'U+{cp:04X}\n\u2192 {display_label}',
                        fontsize=7, rotation=0, labelpad=55, va='center', color=color
                    )
            else:
                ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])

    plt.suptitle('Glyph review  (green = context-labelled, red = needs CNN/manual)', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Step 7b: Manual overrides ─────────────────────────────────────────────
# If the visual review shows a wrong auto-label, add it here.
# Keys must be the exact Python character (copy from the repr column above).
# Values: 'K', 'Q', 'R', 'B', 'N', or '' for pawn.

manual_overrides = {
    # Example:
    # '\u00e9': 'N',   # U+00E9 é was misidentified, actually a Knight
}

context_mapping.update(manual_overrides)
print('Applied', len(manual_overrides), 'manual override(s).')

unlabelled = [ch for ch in candidate_chars if ch not in context_mapping]
print(f'Unlabelled after overrides: {unlabelled}')

## Step 8 — Synthetic Training Data (lichess SVG piece sets)

Only needed when some characters remain unlabelled after Step 6.  Skip this
section entirely if `unlabelled` is empty — the JSON can be exported directly
from `context_mapping`.

In [ ]:
# Class labels for the CNN (index → piece letter / pawn-sentinel).
CNN_CLASSES   = ['K', 'Q', 'R', 'B', 'N', 'P']   # P maps to '' (pawn)
CNN_CLASS_IDX = {p: i for i, p in enumerate(CNN_CLASSES)}

LICHESS_SETS   = ['cburnett', 'merida', 'alpha', 'maestro', 'staunty']
PIECE_SVG_NAMES = ['wK','wQ','wR','wB','wN','wP','bK','bQ','bR','bB','bN','bP']
BASE_URL = 'https://raw.githubusercontent.com/lichess-org/lila/master/public/piece'

# piece_svgs: {set_name: {piece_name: svg_bytes}}
piece_svgs = {s: {} for s in LICHESS_SETS}

def _fetch(set_name, piece_name):
    url = f'{BASE_URL}/{set_name}/{piece_name}.svg'
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            return set_name, piece_name, r.content
    except Exception as e:
        print(f'  \u26a0 {set_name}/{piece_name}: {e}')
    return set_name, piece_name, None

print('Downloading lichess piece sets...')
with ThreadPoolExecutor(max_workers=12) as pool:
    futs = [pool.submit(_fetch, s, p) for s in LICHESS_SETS for p in PIECE_SVG_NAMES]
    for fut in futs:
        sn, pn, data = fut.result()
        if data:
            piece_svgs[sn][pn] = data

for sn in LICHESS_SETS:
    print(f'  {sn}: {len(piece_svgs[sn])}/12')
print('\u2705 Done')

In [ ]:
_svg_cache = {}

def render_piece_svg(piece_name, size=GLYPH_SIZE, bg=255):
    """
    Render a lichess SVG piece on a white background (simulates inline text glyph).
    Returns float32 grayscale array (size, size).
    """
    set_name = next(
        (s for s in LICHESS_SETS if piece_name in piece_svgs[s]), None
    )
    if set_name is None:
        return None
    key = (set_name, piece_name, size)
    if key not in _svg_cache:
        png = cairosvg.svg2png(
            bytestring=piece_svgs[set_name][piece_name],
            output_width=size, output_height=size
        )
        piece_img = Image.open(BytesIO(png)).convert('RGBA')
        bg_img = Image.new('RGBA', (size, size), (bg, bg, bg, 255))
        bg_img = Image.alpha_composite(bg_img, piece_img)
        _svg_cache[key] = np.array(bg_img.convert('L'), dtype=np.float32) / 255.0
    return _svg_cache[key]


def augment_glyph(arr, rng):
    """Augment a (size, size) float32 grayscale array."""
    pil = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8), mode='L')
    pil = ImageEnhance.Brightness(pil).enhance(rng.uniform(0.7, 1.3))
    pil = ImageEnhance.Contrast(pil).enhance(rng.uniform(0.7, 1.4))
    if rng.random() < 0.5:
        pil = pil.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.2, 1.2)))
    arr = np.array(pil, dtype=np.float32) / 255.0
    arr = np.clip(arr + rng.normal(0, 0.025, arr.shape).astype(np.float32), 0, 1)
    # Spatial jitter ±2 px.
    dx, dy = int(rng.integers(-2, 3)), int(rng.integers(-2, 3))
    if dx or dy:
        pil2 = Image.fromarray((arr * 255).astype(np.uint8), mode='L')
        pil2 = pil2.transform(
            pil2.size, Image.AFFINE, (1, 0, dx, 0, 1, dy),
            resample=Image.BILINEAR, fillcolor=int(arr.mean() * 255)
        )
        arr = np.array(pil2, dtype=np.float32) / 255.0
    return arr


# Build synthetic dataset.
AUG_PER_SVG = 40   # augmentations per SVG piece per class

rng = np.random.default_rng(42)
syn_X, syn_y = [], []

# Map CNN classes to SVG piece names.
SVG_FOR_CLASS = {
    'K': ['wK', 'bK'], 'Q': ['wQ', 'bQ'], 'R': ['wR', 'bR'],
    'B': ['wB', 'bB'], 'N': ['wN', 'bN'], 'P': ['wP', 'bP'],
}

for cls, svg_names in SVG_FOR_CLASS.items():
    idx = CNN_CLASS_IDX[cls]
    count = 0
    for sn in LICHESS_SETS:
        for svg_name in svg_names:
            for bg in [255, 220, 180]:   # white, light gray, darker
                base = render_piece_svg(svg_name, bg=bg)
                if base is None:
                    continue
                for _ in range(AUG_PER_SVG):
                    syn_X.append(augment_glyph(base, rng))
                    syn_y.append(idx)
                    count += 1
    print(f'  {cls}: {count} synthetic samples')

syn_X = np.array(syn_X, dtype=np.float32)
syn_y = np.array(syn_y, dtype=np.int64)
print(f'\nSynthetic dataset: {syn_X.shape[0]} samples, {len(CNN_CLASSES)} classes')

## Step 9 — Build Combined Dataset & Train PyTorch CNN

In [ ]:
# Add the labelled PDF glyphs to the dataset (augmented heavily — they're ground truth).
PDF_AUG_FACTOR = 60

rng2 = np.random.default_rng(99)
pdf_X, pdf_y = [], []

for ch, piece in context_mapping.items():
    piece_label = piece if piece else 'P'   # pawn
    if piece_label not in CNN_CLASS_IDX:
        continue
    idx = CNN_CLASS_IDX[piece_label]
    for img in glyph_images.get(ch, []):
        for _ in range(PDF_AUG_FACTOR):
            pdf_X.append(augment_glyph(img, rng2))
            pdf_y.append(idx)

pdf_X = np.array(pdf_X, dtype=np.float32) if pdf_X else np.empty((0, GLYPH_SIZE, GLYPH_SIZE), dtype=np.float32)
pdf_y = np.array(pdf_y, dtype=np.int64)   if pdf_y else np.empty((0,), dtype=np.int64)
print(f'PDF glyph samples (×{PDF_AUG_FACTOR} aug): {len(pdf_X)}')

# Merge.
all_X = np.concatenate([syn_X, pdf_X], axis=0)   # shape (N, 32, 32)
all_y = np.concatenate([syn_y, pdf_y], axis=0)
print(f'Combined: {all_X.shape[0]} samples')

# Add channel dim → (N, 1, 32, 32) for PyTorch.
all_X_t = torch.tensor(all_X[:, None, :, :])   # float32
all_y_t = torch.tensor(all_y)                   # int64


class GlyphDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

full_ds = GlyphDataset(all_X_t, all_y_t)
n_val = max(1, int(len(full_ds) * 0.15))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)
print(f'Train: {n_train}  Val: {n_val}')

In [ ]:
class PieceGlyphCNN(nn.Module):
    """Small CNN for 32×32 grayscale glyph → 6-class piece prediction."""
    def __init__(self, n_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16×16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                             # 8×8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                             # 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PieceGlyphCNN(n_classes=len(CNN_CLASSES)).to(device)
opt    = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
loss_fn = nn.CrossEntropyLoss()

params = sum(p.numel() for p in model.parameters())
print(f'Model: {params:,} parameters  |  device: {device}')


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = total_correct = total_n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            total_loss    += loss.item() * len(y_batch)
            total_correct += (logits.argmax(1) == y_batch).sum().item()
            total_n       += len(y_batch)
    return total_loss / total_n, total_correct / total_n


EPOCHS    = 80
best_val  = 0.0
best_state = None
train_accs, val_accs = [], []

print(f'Training for {EPOCHS} epochs...')
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    sched.step()
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)
    if vl_acc > best_val:
        best_val   = vl_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if ep % 10 == 0 or ep == 1:
        print(f'  ep {ep:3d}  train={tr_acc:.2%}  val={vl_acc:.2%}')

model.load_state_dict(best_state)
print(f'\n\u2705 Best val accuracy: {best_val:.2%}')

In [ ]:
# Training curves.
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(train_accs, label='train')
ax.plot(val_accs,   label='val')
ax.set_title('Glyph CNN accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Confusion matrix.
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X_b, y_b in val_loader:
        preds = model(X_b.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_b.numpy())

cm = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_norm, annot=True, fmt='.0%', xticklabels=CNN_CLASSES,
            yticklabels=CNN_CLASSES, cmap='Blues', ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix (val set)')
plt.tight_layout()
plt.show()

print('Per-class accuracy:')
for i, cls in enumerate(CNN_CLASSES):
    acc = cm_norm[i, i]
    bar = '\u2588' * int(acc * 20)
    print(f'  {cls}: {acc:5.1%}  {bar}')

## Step 10 — Classify Remaining Unlabelled Characters

Run the trained CNN on the collected glyph images for any character that
context-labelling could not resolve.  Majority vote across all crop samples.

In [ ]:
cnn_mapping = {}   # char → piece_letter  (from CNN)

unlabelled = [ch for ch in candidate_chars if ch not in context_mapping]

if not unlabelled:
    print('All characters labelled by context — CNN not needed.')
else:
    model.eval()
    with torch.no_grad():
        for ch in unlabelled:
            imgs = glyph_images.get(ch, [])
            if not imgs:
                print(f'  {repr(ch)}: no glyph images — skipped')
                continue
            batch = torch.tensor(
                np.stack([img[None] for img in imgs], axis=0), dtype=torch.float32
            ).to(device)
            logits = model(batch)              # (n_samples, n_classes)
            probs  = torch.softmax(logits, 1)  # (n_samples, n_classes)
            # Majority vote: pick class with highest mean probability.
            mean_probs = probs.mean(0).cpu().numpy()
            best_cls_idx = int(mean_probs.argmax())
            best_cls     = CNN_CLASSES[best_cls_idx]
            piece_letter = '' if best_cls == 'P' else best_cls
            conf         = mean_probs[best_cls_idx]
            cnn_mapping[ch] = piece_letter
            display = piece_letter if piece_letter else '(pawn)'
            print(f'  {repr(ch)} U+{ord(ch):04X}  \u2192  {display}  (CNN conf: {conf:.1%})')

print(f'\nCNN-labelled: {len(cnn_mapping)}')

## Step 11 — Validate the Mapping

Re-parse the first few pages with the merged mapping and count how many moves
are successfully recovered.

In [ ]:
VALIDATE_PAGES = min(10, len(doc))

final_mapping = {**context_mapping, **cnn_mapping}

def apply_mapping_and_count(page_text, mapping):
    """Parse moves using the mapping; return (moves_found, moves_skipped)."""
    fen = detect_fen(page_text)
    try:
        board = chess.Board(fen) if fen else chess.Board()
    except Exception:
        board = chess.Board()

    found = skipped = 0
    tokens = tokenize(page_text)
    current_move_num = None

    for tok in tokens:
        if _RESULT_RE.match(tok):
            break
        pm = _PURE_NUM_RE.match(tok)
        if pm:
            current_move_num = int(pm.group(1))
            continue
        cm = _COMBINED_RE.match(tok)
        if cm:
            num = int(cm.group(1))
            if current_move_num is None or num >= current_move_num - 1:
                current_move_num = num
                tok = cm.group(3)
        if current_move_num is None:
            continue

        first = tok[0] if tok else ''
        if first in mapping:
            translated = normalize_token(mapping[first] + tok[1:])
        else:
            translated = normalize_token(tok)

        move = try_parse_san(board, translated)
        if move:
            board.push(move)
            found += 1
        elif len(tok) >= 2 and looks_like_san_suffix(tok[1:]):
            skipped += 1

    return found, skipped


print(f'Validation over first {VALIDATE_PAGES} pages:')
print(f'  {"Page":>5}  {"Found":>6}  {"Skipped":>7}  {"Recovery":>9}')
total_found = total_skipped = 0
for pn in range(VALIDATE_PAGES):
    text = doc[pn].get_text()
    found, skipped = apply_mapping_and_count(text, final_mapping)
    total_found   += found
    total_skipped += skipped
    rate = found / (found + skipped) if found + skipped else float('nan')
    print(f'  {pn+1:5d}  {found:6d}  {skipped:7d}  {rate:9.1%}')

total = total_found + total_skipped
overall = total_found / total if total else float('nan')
print(f'\nOverall: {total_found}/{total} moves recovered ({overall:.1%})')

## Step 12 — Export JSON Mapping

The output file `figurine_mapping.json` contains a single object whose keys are
single characters (exactly as returned by pdfrx / PyMuPDF from this PDF) and
whose values are the corresponding English piece letters (`"K"`, `"Q"`, `"R"`,
`"B"`, `"N"`) or `""` for pawn.

### Flutter integration

Pass the mapping into `MoveParser.parse()` via the `fontMap` parameter:

```dart
// Load once at app start (or per-document):
final rawJson = await rootBundle.loadString('assets/figurine_mapping.json');
final fontMap = Map<String, String>.from(jsonDecode(rawJson)['mapping'] as Map);

// Then pass to the parser:
MoveParser.parse(rawText, pageHeight, fontMap: fontMap, ...);
```

In [ ]:
import hashlib

# Build a fingerprint from the PDF to help identify which file the mapping belongs to.
with open(PDF_PATH, 'rb') as f:
    pdf_hash = hashlib.sha256(f.read(65536)).hexdigest()[:12]  # first 64 KB

output = {
    'version': 1,
    'source_pdf': os.path.basename(PDF_PATH),
    'pdf_fingerprint': pdf_hash,
    'labels': {
        # context_labelled: high-confidence, chess-context validated
        ch: {'piece': p, 'method': 'context', 'confidence': confidence.get(ch, 1)}
        for ch, p in context_mapping.items()
    },
    'mapping': {ch: p for ch, p in final_mapping.items()},
}

# Annotate CNN-sourced entries.
for ch, p in cnn_mapping.items():
    output['labels'][ch] = {'piece': p, 'method': 'cnn', 'confidence': None}

OUT_PATH = 'figurine_mapping.json'
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'\u2705 Saved: {OUT_PATH}')
print(f'\nFinal mapping ({len(final_mapping)} entries):')
for ch, piece in sorted(final_mapping.items()):
    cp = ord(ch)
    method = output['labels'].get(ch, {}).get('method', '?')
    display = piece if piece else '(pawn)'
    print(f'  U+{cp:04X}  {repr(ch):8s}  \u2192  {display:8s}  [{method}]')

## Step 13 — Download

In [ ]:
from google.colab import files
files.download(OUT_PATH)
print(f'\nDownloaded {OUT_PATH}')
print('\nNext steps:')
print('  1. Copy figurine_mapping.json to your Flutter assets folder')
print('  2. Declare it in pubspec.yaml under flutter.assets')
print('  3. Load it in the app and pass as fontMap to MoveParser.parse()')